In [ ]:
!pip install -q groq chromadb sentence-transformers

In [ ]:
#   CONFIGURATION — Edit this cell only                    
from datetime import datetime

GROQ_MODEL = "llama-3.1-8b-instant"

_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
MATRIX_FILE = f"literature_matrix_{_stamp}.xlsx"
REVIEW_FILE = f"literature_review_{_stamp}.txt"

In [ ]:
#   Imports & API key setup                          ─
import os, time, re
import requests
import pandas as pd
from datetime import datetime
import xml.etree.ElementTree as ET
from groq import Groq

# Load Groq API key from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    GROQ_API_KEY = UserSecretsClient().get_secret("GROQ_API_KEY")
    print(" Groq API key loaded from Kaggle Secrets")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
    if GROQ_API_KEY:
        print(" Groq API key loaded from environment")
    else:
        print(" No GROQ_API_KEY found.")
        print("   → Get a free key at https://console.groq.com")
        print("   → Add it via Kaggle: Add-ons → Secrets → GROQ_API_KEY")

# Initialize Groq client
groq_client = Groq(api_key=GROQ_API_KEY)
print(f"🤖 LLM model : {GROQ_MODEL}")

✅ Groq API key loaded from Kaggle Secrets
🤖 LLM model : llama-3.1-8b-instant


In [ ]:
#   STEP 1: Fetch papers from arXiv                      

def fetch_arxiv_papers(query, start_year, end_year, max_results):
    """Fetch papers from arXiv API within a year range."""
    print(f"\n🔍 Fetching from arXiv: '{query}' ({start_year}–{end_year})...")

    date_filter  = f"submittedDate:[{start_year}01010000+TO+{end_year}12312359]"
    search_query = f"all:{query.replace(' ', '+')}"

    url = (
        f"https://export.arxiv.org/api/query"
        f"?search_query={search_query}+AND+{date_filter}"
        f"&start=0&max_results={max_results}"
        f"&sortBy=relevance&sortOrder=descending"
    )

    response = requests.get(url, timeout=30)
    if response.status_code != 200:
        print(f"❌ arXiv error: {response.status_code}")
        return []

    ns      = {"atom": "http://www.w3.org/2005/Atom"}
    root    = ET.fromstring(response.content)
    entries = root.findall("atom:entry", ns)

    papers = []
    for entry in entries:
        try:
            year = int(entry.find("atom:published", ns).text[:4])
            if not (start_year <= year <= end_year):
                continue
            authors = [a.find("atom:name", ns).text
                       for a in entry.findall("atom:author", ns)]
            papers.append({
                "title":    entry.find("atom:title", ns).text.strip().replace("\n", " "),
                "authors":  ", ".join(authors[:3]) + (" et al." if len(authors) > 3 else ""),
                "year":     year,
                "abstract": entry.find("atom:summary", ns).text.strip().replace("\n", " "),
                "url":      entry.find("atom:id", ns).text.strip(),
                "source":   "arXiv",
                "journal":  "arXiv preprint"
            })
        except Exception:
            continue

    print(f" Found {len(papers)} papers from arXiv")
    return papers

In [ ]:
#   STEP 2: Fetch papers from CrossRef                    ─

def fetch_crossref_papers(query, start_year, end_year, max_results):
    """Fetch papers from CrossRef API within a year range."""
    print(f"\n🔍 Fetching from CrossRef: '{query}' ({start_year}–{end_year})...")

    params = {
        "query":   query,
        "rows":    max_results,
        "filter":  f"from-pub-date:{start_year},until-pub-date:{end_year}",
        "select":  "title,author,published,abstract,URL,container-title,DOI",
        "mailto":  "research.agent@example.com"
    }

    response = requests.get("https://api.crossref.org/works", params=params, timeout=30)
    if response.status_code != 200:
        print(f"❌ CrossRef error: {response.status_code}")
        return []

    items  = response.json().get("message", {}).get("items", [])
    papers = []

    for item in items:
        try:
            pub  = item.get("published", {}).get("date-parts", [[None]])[0]
            year = pub[0] if pub else None
            if not year or not (start_year <= int(year) <= end_year):
                continue

            raw_authors = item.get("author", [])
            authors     = [
                f"{a.get('given','')} {a.get('family','')}".strip()
                for a in raw_authors[:3]
            ]
            author_str = ", ".join(authors) + (" et al." if len(raw_authors) > 3 else "")

            title_list = item.get("title", [])
            title      = title_list[0] if title_list else "Untitled"

            abstract = item.get("abstract", "Abstract not available.")
            abstract = re.sub(r"<[^>]+>", "", abstract).strip()   # strip JATS XML tags

            journal_list = item.get("container-title", [])
            journal      = journal_list[0] if journal_list else "Unknown journal"

            papers.append({
                "title":    title,
                "authors":  author_str,
                "year":     int(year),
                "abstract": abstract,
                "url":      item.get("URL", ""),
                "source":   "CrossRef",
                "journal":  journal
            })
        except Exception:
            continue

    print(f" Found {len(papers)} papers from CrossRef")
    return papers

In [ ]:
#   STEP 3: Groq LLM helper                          ─

def call_llm(system_prompt, user_prompt, max_tokens=512, retries=3):
    """Call Groq API with retry logic for rate limits."""
    for attempt in range(retries):
        try:
            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt}
                ],
                temperature=0.4,
                max_tokens=max_tokens
            )
            return response.choices[0].message.content.strip()

        except Exception as e:
            err = str(e)
            if "rate_limit" in err.lower() or "429" in err:
                wait = 60   # Groq rate limit resets every minute
                print(f"   ⏳ Rate limit hit — waiting {wait}s (attempt {attempt+1}/{retries})...")
                time.sleep(wait)
            else:
                print(f"   ⚠️  LLM error: {err[:120]}")
                time.sleep(5)

    return "[LLM call failed after retries — check your GROQ_API_KEY]"

In [ ]:
#   STEP 4: Summarize each paper                        

def summarize_paper(paper):
    """Generate a structured 5-field summary for one paper via Groq."""
    system = "You are a research assistant helping write an academic literature review. Be concise and precise."

    user = f"""Summarize this research paper in 5 structured fields.

Title   : {paper['title']}
Authors : {paper['authors']}
Year    : {paper['year']}
Abstract: {paper['abstract'][:1200]}

Respond ONLY in this exact format (each field: 1-2 sentences):
- Core Finding: <what the paper discovered or proved>
- Methodology: <how they conducted the study>
- Key Contribution: <what this adds to the field>
- Limitation: <one key weakness or gap>
- Relevance: <how this relates to '{AREA_OF_INTEREST}'>"""

    return call_llm(system, user, max_tokens=350)


def extract_field(summary, field):
    """Pull a single field out of the structured summary text."""
    match = re.search(rf"{field}:\s*(.+?)(?=\n- |\Z)", summary, re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else ""

In [ ]:
#   STEP 5: Gap analysis + scoring                      ─

def analyze_gaps(papers_with_summaries):
    """Identify research gaps and then score each one for novelty and feasibility."""
    print("\n🔬 Analyzing research gaps...")

    digest = ""
    for i, p in enumerate(papers_with_summaries[:15], 1):
        digest += f"[{i}] {p['title']} ({p['year']})\n"
        digest += f"    Finding   : {p.get('core_finding', 'N/A')}\n"
        digest += f"    Limitation: {p.get('limitation', 'N/A')}\n\n"

    system = "You are an expert academic researcher performing a systematic literature review."

    #   Pass 1: identify gaps                         ─
    user_gaps = f"""I am reviewing literature on: '{AREA_OF_INTEREST}' ({START_YEAR}–{END_YEAR}).

Here are summaries of {len(papers_with_summaries)} papers:
{digest}

Identify and list:
1. Top 5 research gaps (topics not yet studied or understudied)
2. Conflicting findings (where papers disagree)
3. Methodological weaknesses (overused or missing methods)
4. Recommended future research directions

Be specific. This output will appear in a thesis literature review section."""

    raw_gaps = call_llm(system, user_gaps, max_tokens=700)
    print("Gaps identified")

    #   Pass 2: score each gap                         
    print("🏅 Scoring gaps for novelty and feasibility...")

    user_score = f"""You previously identified these research gaps for the topic '{AREA_OF_INTEREST}':

{raw_gaps}

Now score ONLY the 'Top 5 research gaps' section.
For each gap give:
- Gap: <restate the gap in one line>
- Novelty (1-10): <score> — <one sentence reason>
- Feasibility (1-10): <score> — <one sentence reason — consider a student with limited resources>
- Overall (1-10): <average score>
- Verdict: <one sentence on whether a thesis student should pursue this>

Format each gap as a numbered block. Be direct and practical."""

    scored_gaps = call_llm(system, user_score, max_tokens=700)
    print("Gaps scored")

    #   Combine both passes                          ─
    full_output = (
        "RESEARCH GAPS & ANALYSIS\n" + "-" * 40 + "\n"
        + raw_gaps
        + "\n\n" + "=" * 40 + "\n"
        + "GAP SCORING (Novelty · Feasibility · Verdict)\n" + "-" * 40 + "\n"
        + scored_gaps
    )
    return full_output

In [ ]:
#   STEP 6: Write the literature review                    

def write_literature_review(papers_with_summaries, gap_analysis):
    """Generate a full structured literature review via Groq."""
    print("\n✍️  Writing literature review...")

    papers_text = ""
    for p in papers_with_summaries[:12]:
        papers_text += (
            f"- {p['authors']} ({p['year']}). {p['title']}.\n"
            f"  Finding: {p.get('core_finding', p['abstract'][:150])}\n\n"
        )

    system = "You are an expert academic writer. Write in formal, scholarly English suitable for a university thesis."

    user = f"""Write a literature review section for a thesis on: '{AREA_OF_INTEREST}'.

Sources ({len(papers_with_summaries)} papers):
{papers_text}

Research gaps identified:
{gap_analysis[:600]}

Structure the review with these 5 sections:
1. Introduction — scope and purpose of this review
2. Thematic analysis — group papers by theme, do NOT just list them one by one
3. Evolution of the field ({START_YEAR}–{END_YEAR}) — how research has progressed
4. Research gaps and limitations — what is still unknown
5. Conclusion — summary and link to the thesis research question

Cite as (Author, Year). Write ~500 words. Academic tone only."""

    return call_llm(system, user, max_tokens=1000)

In [ ]:
#   STEP 7: Export literature matrix to Excel                 ─

def export_matrix(papers_with_summaries, filename):
    """Save all paper data + LLM summaries as a formatted Excel file with wrapped cells."""
    print(f"\n📊 Exporting literature matrix → {filename}...")

    rows = []
    for i, p in enumerate(papers_with_summaries, 1):
        rows.append({
            "#":                i,
            "Title":            p.get("title", ""),
            "Authors":          p.get("authors", ""),
            "Year":             p.get("year", ""),
            "Journal/Source":   p.get("journal", ""),
            "Source DB":        p.get("source", ""),
            "Core Finding":     p.get("core_finding", ""),
            "Methodology":      p.get("methodology", ""),
            "Key Contribution": p.get("contribution", ""),
            "Limitation":       p.get("limitation", ""),
            "Relevance":        p.get("relevance", ""),
            "URL":              p.get("url", "")
        })

    df = pd.DataFrame(rows)

    from openpyxl.styles import PatternFill, Font, Alignment
    from openpyxl.utils import get_column_letter

    with pd.ExcelWriter(filename, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="Literature Matrix")
        ws = writer.sheets["Literature Matrix"]

        # Style header row
        header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = Font(color="FFFFFF", bold=True)
            cell.alignment = Alignment(wrap_text=True, vertical="center")

        # Column width rules — wide for text-heavy cols, narrow for short ones
        col_widths = {
            "#": 5, "Title": 40, "Authors": 25, "Year": 8,
            "Journal/Source": 28, "Source DB": 12,
            "Core Finding": 45, "Methodology": 35,
            "Key Contribution": 45, "Limitation": 35,
            "Relevance": 40, "URL": 35
        }

        for col_idx, col_name in enumerate(df.columns, 1):
            col_letter = get_column_letter(col_idx)
            ws.column_dimensions[col_letter].width = col_widths.get(col_name, 20)

        # Wrap text + auto row height for every data row
        for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
            max_lines = 1
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical="top")
                if cell.value:
                    text   = str(cell.value)
                    col_w  = col_widths.get(df.columns[cell.column - 1], 20)
                    # estimate line count: chars / col_width + explicit newlines
                    lines  = max(1, -(-len(text) // max(col_w, 1))) + text.count("\n")
                    max_lines = max(max_lines, lines)
            # Set row height: ~15pt per line, min 20, max 200
            ws.row_dimensions[row[0].row].height = min(max(max_lines * 15, 20), 200)

    print(f" Saved {len(rows)} papers → {filename}")
    return df

In [ ]:
#   STEP 8: Main agent runner                         ─

def run_agent():
    print("=" * 60)
    print("📚 LITERATURE REVIEW AI AGENT")
    print("=" * 60)
    print(f"Topic      : {AREA_OF_INTEREST}")
    print(f"Years      : {START_YEAR} – {END_YEAR}")
    print(f"Papers/run : {PAPERS_PER_RUN}")
    print(f"LLM        : Groq / {GROQ_MODEL}")
    print("=" * 60)

    #   1. Fetch papers                             
    half            = PAPERS_PER_RUN // 2
    arxiv_papers    = fetch_arxiv_papers(AREA_OF_INTEREST, START_YEAR, END_YEAR, half)
    crossref_papers = fetch_crossref_papers(AREA_OF_INTEREST, START_YEAR, END_YEAR, PAPERS_PER_RUN - half)

    # Deduplicate by normalised title
    seen, unique_papers = set(), []
    for p in arxiv_papers + crossref_papers:
        key = re.sub(r"[^a-z0-9]", "", p["title"].lower())
        if key not in seen:
            seen.add(key)
            unique_papers.append(p)

    print(f"\n📑 Total unique papers: {len(unique_papers)}")
    if not unique_papers:
        print("❌ No papers found. Try a broader topic or wider year range.")
        return

    #   2. Summarize each paper                         
    print("\n🤖 Summarizing papers with Groq LLM...")
    papers_with_summaries = []

    for i, paper in enumerate(unique_papers, 1):
        print(f"   [{i}/{len(unique_papers)}] {paper['title'][:72]}...")
        summary = summarize_paper(paper)
        paper["summary"]      = summary
        paper["core_finding"] = extract_field(summary, "Core Finding")
        paper["methodology"]  = extract_field(summary, "Methodology")
        paper["contribution"] = extract_field(summary, "Key Contribution")
        paper["limitation"]   = extract_field(summary, "Limitation")
        paper["relevance"]    = extract_field(summary, "Relevance")
        papers_with_summaries.append(paper)
        time.sleep(2)   # stay within Groq's 30 req/min free limit

    #   3. Gap analysis                             
    gap_analysis = analyze_gaps(papers_with_summaries)

    #   4. Written literature review                      
    lit_review = write_literature_review(papers_with_summaries, gap_analysis)

    #   5. Export Excel matrix                         ─
    df = export_matrix(papers_with_summaries, MATRIX_FILE)

    #   6. Save full text output                        ─
    with open(REVIEW_FILE, "w", encoding="utf-8") as f:
        f.write("LITERATURE REVIEW\n")
        f.write(f"Topic    : {AREA_OF_INTEREST}\n")
        f.write(f"Years    : {START_YEAR}–{END_YEAR}\n")
        f.write(f"Model    : Groq / {GROQ_MODEL}\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
        f.write("=" * 60 + "\n\n")

        f.write("WRITTEN LITERATURE REVIEW\n" + "-" * 40 + "\n")
        f.write(lit_review + "\n\n")

        f.write("=" * 60 + "\n\n")
        f.write("RESEARCH GAPS & FUTURE DIRECTIONS\n" + "-" * 40 + "\n")
        f.write(gap_analysis + "\n\n")

        f.write("=" * 60 + "\n\n")
        f.write("INDIVIDUAL PAPER SUMMARIES\n" + "-" * 40 + "\n")
        for p in papers_with_summaries:
            f.write(f"\n[{p['year']}] {p['title']}\n")
            f.write(f"Authors : {p['authors']}\n")
            f.write(f"Source  : {p['source']} | {p['journal']}\n")
            f.write(f"URL     : {p['url']}\n")
            f.write(f"Summary :\n{p['summary']}\n")
            f.write("-" * 40 + "\n")

    print(f"✅ Saved: {REVIEW_FILE}")

    #   7. Preview in notebook                         ─
    print("\n" + "=" * 60)
    print("📋 LITERATURE MATRIX PREVIEW (first 5 papers)")
    print("=" * 60)
    display(df[["#", "Title", "Authors", "Year", "Core Finding"]].head(5))

    print("\n" + "=" * 60)
    print("🔬 RESEARCH GAPS")
    print("=" * 60)
    print(gap_analysis)

    print("\n" + "=" * 60)
    print("✍️  LITERATURE REVIEW (first 1500 chars)")
    print("=" * 60)
    print(lit_review[:1500])

    print("\n" + "=" * 60)
    print("🎉 AGENT COMPLETE")
    print(f"   📊 Matrix  → {MATRIX_FILE}")
    print(f"   📝 Review  → {REVIEW_FILE}")
    print("=" * 60)




In [ ]:
#   UI: Interactive input form                         
import ipywidgets as widgets
from IPython.display import display, clear_output

topic_input = widgets.Text(
    value="large language models in education",
    description="Topic:",
    layout=widgets.Layout(width="500px"),
    style={"description_width": "80px"}
)
start_input = widgets.IntText(
    value=2020, description="Start year:",
    layout=widgets.Layout(width="200px"),
    style={"description_width": "80px"}
)
end_input = widgets.IntText(
    value=2024, description="End year:",
    layout=widgets.Layout(width="200px"),
    style={"description_width": "80px"}
)
papers_input = widgets.IntSlider(
    value=10, min=5, max=30, step=1,
    description="Max papers:",
    layout=widgets.Layout(width="400px"),
    style={"description_width": "80px"}
)
run_button = widgets.Button(
    description="▶ Run Agent",
    button_style="success",
    layout=widgets.Layout(width="160px", height="36px")
)
out = widgets.Output()

def on_run_clicked(b):
    # Push UI values into the global config variables
    global AREA_OF_INTEREST, START_YEAR, END_YEAR, PAPERS_PER_RUN, MATRIX_FILE, REVIEW_FILE
    AREA_OF_INTEREST = topic_input.value.strip()
    START_YEAR       = start_input.value
    END_YEAR         = end_input.value
    PAPERS_PER_RUN   = papers_input.value
    _stamp           = datetime.now().strftime("%Y%m%d_%H%M%S")
    MATRIX_FILE      = f"literature_matrix_{_stamp}.xlsx"
    REVIEW_FILE      = f"literature_review_{_stamp}.txt"

    with out:
        clear_output()
        run_agent()

run_button.on_click(on_run_clicked)

display(
    widgets.VBox([
        widgets.HTML("<h3>📚 Literature Review Agent</h3>"),
        topic_input,
        widgets.HBox([start_input, end_input]),
        papers_input,
        run_button,
        out
    ])
)